# LIBERO eval — 노드 2 (**GPU 4장**)

**이미 학습된 4개 모델**(`acm` · `act` · `bimamba` · `bimamba_s7`)을 LIBERO-10 에서 eval (학습 안 함).
노드 2대 = GPU **4 + 4**. 이 노트북은 **노드 2** 몫(8 eval)만. 나머지 8개는 다른 노드 → `eval_node1_4gpu`.

- 전체 = 4모델 × 4 seed(0-3) = **16 eval**, GPU 비(4:4)로 8/8 분배.
- 150k 체크포인트 × **500ep**, action(.pt) 기록 → jerk/LDJ/SPARC/SignFlip.
- 체크포인트 없거나 이미 끝난 eval 은 **자동 skip** → 재실행 안전.
- ⚠️ **LIBERO 시뮬 필요**. 결과 = `eval_clean/libero_10/…` → 리포트 자동 pooled.
- `bimamba_s7` 은 아직 rename 전 폴더명 그대로 사용(나중에 `utils/0m` 돌리면 mosaic 으로 통일).


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

## 1) 세팅 + 이 노드가 돌릴 eval 목록
체크포인트 상태(`OK 150k` / `❌ 없음` / `⚠ 최근접`)를 같이 확인한다.


In [ ]:
# ── 실제 학습된 모델 = train/libero_10/<이 이름>/seed0..3 폴더명 ──
MODELS = ['acm', 'act', 'bimamba', 'bimamba_s7']   # 4개 모델
SEEDS  = [0, 1, 2, 3]
N_EP   = 500
STEP   = cf.CKPT_STEP                                # 150k 체크포인트
GPU_COUNTS = [4, 4]                                  # 노드 2대 = GPU 4 + 4

# 비표준 태그(bimamba_s7 등)를 폴더명 그대로 등록 → train_dir/eval_dir 가 찾게
for m in MODELS:
    cf.v23.MODEL_DIR_NAMES.setdefault(m, m)

ALL = [(m, s) for s in SEEDS for m in MODELS]        # 4모델 × 4seed = 16 eval
NODE_IDX = 1
MINE = cf.split_by_gpu(ALL, GPU_COUNTS)[NODE_IDX]
gpus = cf.available_gpus()[:GPU_COUNTS[NODE_IDX]]
print(f'노드 {NODE_IDX + 1} (4GPU) | GPU {gpus} | {len(MINE)} eval × {N_EP}ep')
print('두 노드 분배:', [len(x) for x in cf.split_by_gpu(ALL, GPU_COUNTS)], '\n')
for m, s in MINE:
    got = cf.resolved_ckpt_step(m, s, step=STEP)
    flag = 'OK 150k' if got == STEP else ('❌ ckpt 없음' if got is None else f'⚠ 최근접 {got:,}')
    print(f'   {m:12} seed{s}   {flag}')

## 2) 실행 — 500ep, GPU 하나당 eval 하나(OOM 방지)


In [ ]:
cf.run_libero_eval_jobs(MINE, gpus=gpus, n_episodes=N_EP, step=STEP)

## 3) 결과 (SR + 영상 수)


In [ ]:
import glob
print(f"{'MODEL':<14}{'SEED':>5}{'SR':>9}{'VIDEOS':>8}")
print('-' * 36)
for m, s in MINE:
    st = cf.get_eval_status(m, s, 'libero_10')
    sr = f"{st['sr']*100:.1f}%" if st['sr'] is not None else '-'
    vids = glob.glob(str(cf.eval_clean_dir(m, s, 'libero_10') / '**' / '*.mp4'), recursive=True)
    print(f'{m:<14}{s:>5}{sr:>9}{len(vids):>8}')